# Best and Worst GAX Visualization + Safe Cleanup

This notebook:
1. Mounts Google Drive.
2. Lets you set paths in one configuration cell.
3. Loads the cheating score CSV.
4. Selects the best and worst examples based on `cheating_score`.
5. Generates 3-panel visualizations:
   - Original X-ray
   - Segmented lung mask
   - GAX heatmap overlay
6. Saves outputs in a new folder.
7. Includes a separate safe deletion cell for GAX outputs after verification.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration Cell

Edit only this cell for each model.

Example for model v7:
- `MODEL_NAME = "v7"`
- `GAX_DIR = "/content/drive/Shareddrives/v7_gax_outputs/outputs"`
- `CSV_PATH = ".../cheating_scores.csv"`
- `SAVE_ROOT = ".../gax_selected_visuals"`


In [ ]:
# =========================
# USER CONFIGURATION
# =========================

MODEL_NAME = "v7"

# Path to cheating_scores.csv
CSV_PATH = "/content/drive/MyDrive/thesis_results/v7/cheating_scores.csv"

# Path to GAX outputs folder
GAX_DIR = "/content/drive/Shareddrives/v7_gax_outputs/outputs"

# Original and mask test image folders
ORIGINAL_DIR = "/content/jpeg_dataset/test"
MASK_DIR = "/content/masked_dataset/test"

# Root folder where visualizations will be saved
# A new folder for the model will be created automatically inside this folder.
SAVE_ROOT = "/content/drive/MyDrive/gax_selected_visuals"

# Number of best and worst examples to visualize
N_WORST = 1
N_BEST = 1

# If True, saves a CSV summary of selected best/worst images
SAVE_SUMMARY_CSV = True

## Import Libraries and Validate Paths

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

def validate_paths(csv_path, gax_dir, original_dir, mask_dir):
    """Checks if required paths exist before running visualization."""

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    if not os.path.exists(gax_dir):
        raise FileNotFoundError(f"GAX output directory not found: {gax_dir}")

    if not os.path.exists(original_dir):
        raise FileNotFoundError(f"Original image directory not found: {original_dir}")

    if not os.path.exists(mask_dir):
        raise FileNotFoundError(f"Mask image directory not found: {mask_dir}")

    print("All required paths found.")

# Create separate save folder for the selected model
SAVE_DIR = os.path.join(SAVE_ROOT, MODEL_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

validate_paths(CSV_PATH, GAX_DIR, ORIGINAL_DIR, MASK_DIR)

print(f"Model: {MODEL_NAME}")
print(f"Visualizations will be saved to: {SAVE_DIR}")

## Define Helper Functions

In [ ]:
def get_positive_heatmap(raw_mask):
    """Extract positive attributions and normalize them for visual overlay."""

    if raw_mask.ndim == 3 and raw_mask.shape[-1] in [1, 3]:
        h_sum = np.sum(raw_mask, axis=2)
    else:
        h_sum = raw_mask

    positive_attr = np.maximum(h_sum, 0)

    if np.max(positive_attr) > 0:
        positive_attr = positive_attr / np.max(positive_attr)

    return positive_attr


def find_gax_file(gax_dir, img_name):
    """Find matching GAX file using mult or sum naming convention."""

    possible_paths = [
        os.path.join(gax_dir, f"op.{img_name}.test.mult.npy"),
        os.path.join(gax_dir, f"op.{img_name}.test.sum.npy")
    ]

    for path in possible_paths:
        if os.path.exists(path):
            return path

    return None


def generate_visual(row, prefix_name, model_name, gax_dir, output_dir):
    """Generate and save one 3-panel visual proof."""

    img_name = row["image_name"]
    true_class = row["true_class"]
    score = row["cheating_score"]

    orig_path = os.path.join(ORIGINAL_DIR, true_class, img_name)
    mask_path = os.path.join(MASK_DIR, true_class, img_name)
    gax_path = find_gax_file(gax_dir, img_name)

    if not os.path.exists(orig_path):
        print(f"[SKIPPED] Missing original image: {orig_path}")
        return False

    if not os.path.exists(mask_path):
        print(f"[SKIPPED] Missing mask image: {mask_path}")
        return False

    if gax_path is None:
        print(f"[SKIPPED] Missing GAX file for: {img_name}")
        return False

    # Load original image
    orig_img = cv2.imread(orig_path)
    orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
    orig_img = cv2.resize(orig_img, (224, 224))

    # Load mask image
    mask_img = cv2.imread(mask_path)
    mask_img = cv2.cvtColor(mask_img, cv2.COLOR_BGR2RGB)
    mask_img = cv2.resize(mask_img, (224, 224))

    # Load GAX heatmap
    gax_data = np.load(gax_path)
    heatmap_2d = get_positive_heatmap(gax_data[-1])

    # Create 3-panel visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    fig.suptitle(
        f"{model_name.upper()} {prefix_name.upper()} Example | "
        f"Cheating Score: {score:.1%} | Class: {true_class}",
        fontsize=15,
        fontweight="bold"
    )

    axes[0].imshow(orig_img)
    axes[0].set_title("Original X-Ray")
    axes[0].axis("off")

    axes[1].imshow(mask_img)
    axes[1].set_title("Segmented Lung Mask")
    axes[1].axis("off")

    axes[2].imshow(orig_img)
    heatmap_overlay = np.ma.masked_where(heatmap_2d < 0.15, heatmap_2d)
    axes[2].imshow(heatmap_overlay, cmap="jet", alpha=0.55)
    axes[2].set_title("GAX Heatmap Overlay")
    axes[2].axis("off")

    plt.tight_layout()

    safe_img_name = os.path.splitext(img_name)[0]
    save_path = os.path.join(
        output_dir,
        f"{model_name}_{prefix_name}_{safe_img_name}.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"[SAVED] {save_path}")
    return True

## Load CSV and Select Best/Worst Examples

In [ ]:
df = pd.read_csv(CSV_PATH)

required_columns = ["image_name", "true_class", "cheating_score"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns in CSV: {missing_columns}")

df_sorted = df.sort_values(by="cheating_score", ascending=False)

worst_cases = df_sorted.head(N_WORST)
best_cases = df_sorted.tail(N_BEST)

print("Worst case/s:")
display(worst_cases)

print("\nBest case/s:")
display(best_cases)

## Generate Best and Worst Visualizations

In [ ]:
summary_rows = []

print(f"Generating worst example/s for {MODEL_NAME}...")
for _, row in worst_cases.iterrows():
    success = generate_visual(row, "worst", MODEL_NAME, GAX_DIR, SAVE_DIR)
    if success:
        summary_rows.append({
            "model": MODEL_NAME,
            "type": "worst",
            "image_name": row["image_name"],
            "true_class": row["true_class"],
            "cheating_score": row["cheating_score"]
        })

print(f"\nGenerating best example/s for {MODEL_NAME}...")
for _, row in best_cases.iterrows():
    success = generate_visual(row, "best", MODEL_NAME, GAX_DIR, SAVE_DIR)
    if success:
        summary_rows.append({
            "model": MODEL_NAME,
            "type": "best",
            "image_name": row["image_name"],
            "true_class": row["true_class"],
            "cheating_score": row["cheating_score"]
        })

summary_df = pd.DataFrame(summary_rows)

if SAVE_SUMMARY_CSV:
    summary_csv_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_best_worst_summary.csv")
    summary_df.to_csv(summary_csv_path, index=False)
    print(f"\nSummary CSV saved to: {summary_csv_path}")

display(summary_df)

## Check Saved Files Before Deleting Anything

In [ ]:
print(f"Files saved in: {SAVE_DIR}")

for file in os.listdir(SAVE_DIR):
    print(file)

## Optional: Preview Saved Images

In [ ]:
from IPython.display import Image, display

saved_images = [
    os.path.join(SAVE_DIR, file)
    for file in os.listdir(SAVE_DIR)
    if file.lower().endswith(".png")
]

for img_path in saved_images:
    print(img_path)
    display(Image(filename=img_path))

# Safe Deletion Section

Only run this after confirming that:
1. Your best/worst visualizations were saved correctly.
2. Your summary CSV was saved correctly.
3. You no longer need the full GAX `.npy` outputs.

By default, deletion is disabled.


In [ ]:
DELETE_GAX_OUTPUTS = False  # Change to True only when you are 100% ready

if DELETE_GAX_OUTPUTS:
    if os.path.exists(GAX_DIR):
        shutil.rmtree(GAX_DIR)
        print(f"[DELETED] GAX output directory deleted: {GAX_DIR}")
    else:
        print(f"[SKIPPED] GAX directory not found: {GAX_DIR}")
else:
    print("Deletion skipped. Set DELETE_GAX_OUTPUTS = True when you are ready.")

## Optional: Multi-Model Configuration Template

Use this only if you want to process multiple models in one run by manually listing their CSV and GAX paths.


In [ ]:
# MULTI_MODEL_CONFIGS = [
#     {
#         "model_name": "v2",
#         "csv_path": "/content/drive/MyDrive/thesis_results/v2/cheating_scores.csv",
#         "gax_dir": "/content/drive/Shareddrives/v2_gax_outputs/outputs"
#     },
#     {
#         "model_name": "v3",
#         "csv_path": "/content/drive/MyDrive/thesis_results/v3/cheating_scores.csv",
#         "gax_dir": "/content/drive/Shareddrives/v3_gax_outputs/outputs"
#     },
# ]

# for config in MULTI_MODEL_CONFIGS:
#     model_name = config["model_name"]
#     csv_path = config["csv_path"]
#     gax_dir = config["gax_dir"]
#     save_dir = os.path.join(SAVE_ROOT, model_name)
#     os.makedirs(save_dir, exist_ok=True)
#
#     if not os.path.exists(csv_path):
#         print(f"[SKIPPED] Missing CSV for {model_name}: {csv_path}")
#         continue
#
#     if not os.path.exists(gax_dir):
#         print(f"[SKIPPED] Missing GAX folder for {model_name}: {gax_dir}")
#         continue
#
#     temp_df = pd.read_csv(csv_path)
#     temp_df_sorted = temp_df.sort_values(by="cheating_score", ascending=False)
#
#     temp_worst = temp_df_sorted.head(N_WORST)
#     temp_best = temp_df_sorted.tail(N_BEST)
#
#     for _, row in temp_worst.iterrows():
#         generate_visual(row, "worst", model_name, gax_dir, save_dir)
#
#     for _, row in temp_best.iterrows():
#         generate_visual(row, "best", model_name, gax_dir, save_dir)
#
#     print(f"[DONE] {model_name}")